# AllyVision: object detection and depth on uploaded video

Run in Google Colab and enable a GPU runtime for faster processing. Upload one video and download the annotated MP4 after processing.

**Depth limitation:** the selected checkpoint is configured for metric indoor depth. Metric configuration does not guarantee calibration for every camera, scene or object; it is intended for indoor use and is not validated for outdoor distances. The notebook reports a metric unit only when the loaded model configuration explicitly exposes its metric-depth flag. Treat estimates as perception hints, never guaranteed distances.


## 1. Installation and Imports

In [ ]:
# Colab runtime: install once per fresh runtime.
%pip -q install -U ultralytics transformers

In [ ]:
import math
from pathlib import Path
from tempfile import mkdtemp

import cv2
import numpy as np
import torch
from PIL import Image
from ultralytics import YOLO
from transformers import pipeline

try:
    from google.colab import files as colab_files
except ModuleNotFoundError:
    colab_files = None


## 2. Configuration

In [ ]:
YOLO_WEIGHTS = "yolo11n.pt"
DEPTH_MODEL_ID = "depth-anything/Depth-Anything-V2-Metric-Indoor-Small-hf"
CONFIDENCE_THRESHOLD = 0.25  # configurable; changing it filters boxes, not model knowledge
FRAME_SKIP = 1               # process each frame; increase to 2/3 to reduce workload
DEPTH_REGION_INNER_FRACTION = 0.50  # central half of each box in both dimensions
DEVICE = 0 if torch.cuda.is_available() else -1


## 3. Model Loading
Run once. YOLO uses its pretrained class vocabulary (COCO for the default `yolo11n.pt` weights); objects outside that vocabulary cannot be named by this model. A low confidence score or no detection can also reflect poor visibility/model error, not proof that an object is absent.

In [ ]:
detector = YOLO(YOLO_WEIGHTS)
depth_estimator = pipeline("depth-estimation", model=DEPTH_MODEL_ID, device=DEVICE)

# Pipeline model configuration diagnostic.
config = getattr(depth_estimator.model, "config", None)
config_dict = config.to_dict() if config is not None and hasattr(config, "to_dict") else {}
metric_flag = config_dict.get(" ZoeDepth".strip(), None)  # harmless fallback for config variants
metric_flag = config_dict.get("metric_depth", getattr(config, "metric_depth", None))
DEPTH_UNIT = "m (model configured metric; approximate)" if metric_flag is True else "unverified"
print("YOLO supported classes:", len(detector.names))
print("Depth model:", DEPTH_MODEL_ID)
print("Depth config metric_depth:", metric_flag, "=> unit:", DEPTH_UNIT)
print("Runtime device:", "GPU" if DEVICE >= 0 else "CPU")


## 4. Choose a Video
Upload one video in Colab. In local Jupyter, set `VIDEO_PATH` to the file on your machine.

In [ ]:
VIDEO_PATH = Path("input.mp4")  # change this when running locally

if colab_files is not None:
    uploaded = colab_files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one video.")
    VIDEO_PATH = Path(next(iter(uploaded)))
    del uploaded

if not VIDEO_PATH.is_file():
    raise FileNotFoundError(f"Could not find video: {VIDEO_PATH}")


## 5. Object Detection and 6. Depth Estimation

In [ ]:
def _depth_array(pil_image):
    output = depth_estimator(pil_image)
    depth = output.get("predicted_depth")
    if depth is None:
        raise ValueError("Depth pipeline returned no predicted_depth")
    if isinstance(depth, Image.Image):
        depth = np.asarray(depth).astype(np.float32)
    elif torch.is_tensor(depth):
        depth = depth.detach().float().cpu().numpy()
    else:
        depth = np.asarray(depth, dtype=np.float32)
    depth = np.squeeze(depth)
    if depth.ndim != 2 or not depth.size:
        raise ValueError(f"Unexpected depth output shape: {depth.shape}")
    return depth

def _robust_region_depth(depth_map, bbox):
    h,w = depth_map.shape
    x1,y1,x2,y2 = bbox
    bw=max(1,x2-x1); bh=max(1,y2-y1)
    mx=(1-DEPTH_REGION_INNER_FRACTION)*bw/2; my=(1-DEPTH_REGION_INNER_FRACTION)*bh/2
    xa=max(0,min(w,int(x1+mx))); xb=max(0,min(w,int(x2-mx)))
    ya=max(0,min(h,int(y1+my))); yb=max(0,min(h,int(y2-my)))
    region=depth_map[ya:yb,xa:xb]
    valid=region[np.isfinite(region)] if region.size else np.array([])
    return float(np.median(valid)) if valid.size else None


## 7. Combined Frame Processing

In [ ]:
def process_frame(frame):
    """Run detector and depth once each; return structured objects and annotated BGR frame."""
    if frame is None or frame.size == 0:
        raise ValueError("Empty video frame")
    h,w=frame.shape[:2]
    pil=Image.fromarray(cv2.cvtColor(frame,cv2.COLOR_BGR2RGB))
    depth_error=None
    try:
        raw_depth=_depth_array(pil)
        depth_map=cv2.resize(raw_depth,(w,h),interpolation=cv2.INTER_LINEAR)
    except Exception as exc:
        depth_map=None; depth_error=str(exc)
    result=detector.predict(frame,conf=CONFIDENCE_THRESHOLD,verbose=False)[0]
    annotated=frame.copy(); objects=[]
    names=detector.names
    for box in result.boxes:
        x1,y1,x2,y2=map(int,box.xyxy[0].detach().cpu().tolist())
        x1=max(0,min(w-1,x1)); x2=max(0,min(w,x2)); y1=max(0,min(h-1,y1)); y2=max(0,min(h,y2))
        class_id=int(box.cls[0].item()); label=names.get(class_id,f"unknown_class_{class_id}") if isinstance(names,dict) else names[class_id]
        confidence=float(box.conf[0].item()); center=[(x1+x2)//2,(y1+y2)//2]
        value=_robust_region_depth(depth_map,[x1,y1,x2,y2]) if depth_map is not None else None
        obj={"class_name":str(label),"confidence":confidence,"bbox":[x1,y1,x2,y2],"center":center,"depth":value,"depth_unit":DEPTH_UNIT if value is not None else "unavailable"}
        objects.append(obj)
        text=f"{label} {confidence:.2f} | depth {value:.2f} {DEPTH_UNIT}" if value is not None else f"{label} {confidence:.2f} | depth unavailable"
        cv2.rectangle(annotated,(x1,y1),(x2,y2),(40,220,40),2)
        cv2.putText(annotated,text,(x1,max(20,y1-8)),cv2.FONT_HERSHEY_SIMPLEX,0.48,(40,220,40),2,cv2.LINE_AA)
    return {"objects":objects,"depth_unit":DEPTH_UNIT,"depth_error":depth_error},annotated


## 8. Process the Video
YOLO detects objects on each processed frame. `FRAME_SKIP` controls how many frames are sampled. The annotated MP4 preserves the source playback speed and contains video frames without audio.

In [ ]:
if FRAME_SKIP < 1:
    raise ValueError("FRAME_SKIP must be at least 1.")

output_path = Path(mkdtemp(prefix="allyvision-")) / f"{VIDEO_PATH.stem}_annotated.mp4"
capture = cv2.VideoCapture(str(VIDEO_PATH))
writer = None
frame_count = 0
processed_count = 0

try:
    if not capture.isOpened():
        raise RuntimeError(f"Could not open video: {VIDEO_PATH}")
    source_fps = capture.get(cv2.CAP_PROP_FPS)
    if not math.isfinite(source_fps) or source_fps <= 0:
        source_fps = 25.0

    while True:
        ok, frame = capture.read()
        if not ok:
            break
        frame_count += 1
        if (frame_count - 1) % FRAME_SKIP:
            continue

        _, annotated = process_frame(frame)
        if writer is None:
            height, width = annotated.shape[:2]
            writer = cv2.VideoWriter(
                str(output_path), cv2.VideoWriter_fourcc(*"mp4v"),
                source_fps / FRAME_SKIP, (width, height),
            )
            if not writer.isOpened():
                raise RuntimeError(f"Could not create output video: {output_path}")
        writer.write(annotated)
        processed_count += 1

    if not processed_count:
        raise RuntimeError(f"No readable frames in {VIDEO_PATH}")
finally:
    capture.release()
    if writer is not None:
        writer.release()

print(f"Saved {processed_count} annotated frames to {output_path}")
if colab_files is not None:
    colab_files.download(str(output_path))


## 9. Testing and Diagnostics
Open the exported MP4 and check for object labels, boxes, and depth values. If processing is slow, enable a GPU in Colab or increase `FRAME_SKIP` in step 2.